# scVI Integration with HCA Lung Reference (v2 — fixed)

Jointly trains scVI on in-house data (fused, parental, resistant) and a public HCA lung
reference dataset. The batch covariate is `source` (in_house vs public_HCA).

**Changes from the original notebook**, made to diagnose/improve low overlap after
1000-epoch training:
1. `cell_type` (and `assay`) from HCA are preserved through the concat (previously
   silently dropped by the inner join on `.obs`).
2. `assay` is added as an extra categorical covariate — the HCA pull mixes multiple
   sequencing technologies, which was previously unmodeled technical variation sitting
   on top of the in_house/public_HCA batch effect.
3. `n_latent` reduced to 30 (128 was too high — leaves room to encode batch-specific
   structure instead of removing it) and `max_epochs` is now set from scvi-tools'
   own sizing heuristic with early stopping, instead of a flat 1000, with ELBO curves
   plotted to confirm convergence rather than assuming more epochs = better mixing.
4. New section: per-cell and per-cluster **mixing diagnostics** to identify which
   in-house cells are actually overlapping with the HCA reference, and which HCA
   cell types they resemble.

**Workflow:**
1. Load in-house 10x data
2. Download HCA lung data via CellxGene Census
3. Verify raw counts in HCA data
4. Tag + concatenate (preserving `cell_type`/`assay`)
5. QC filter
6. HVG selection with `batch_key='source'`
7. scVI with `batch_key='source'`, `categorical_covariate_keys=['sample', 'assay']`
8. Embedding, clustering, visualization
9. Mixing diagnostics (which cells overlap, and with what)
10. DE between in-house sample groups (restricted to in-house cells)
11. Cell type composition / in-house-only UMAP

## Imports

In [1]:
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import scvi
import cellxgene_census  # pip install cellxgene-census
import matplotlib.pyplot as plt

sc.set_figure_params(figsize=(5, 5))
%config InlineBackend.print_figure_kwargs={'facecolor': 'w'}
%config InlineBackend.figure_format='retina'

## 1. Load In-House Data

In [2]:
adata_fus = sc.read_10x_mtx('../fused/', var_names='gene_symbols', cache=True)
adata_fus.obs['sample'] = 'fused'
adata_fus.obs.index = [f'F_{bc}' for bc in adata_fus.obs.index]

adata_par = sc.read_10x_mtx('../parental/', var_names='gene_symbols', cache=True)
adata_par.obs['sample'] = 'parental'
adata_par.obs.index = [f'P_{bc}' for bc in adata_par.obs.index]

adata_res = sc.read_10x_mtx('../resistant/', var_names='gene_symbols', cache=True)
adata_res.obs['sample'] = 'resistant'
adata_res.obs.index = [f'R_{bc}' for bc in adata_res.obs.index]

for adata in [adata_fus, adata_par, adata_res]:
    adata.var_names_make_unique()
    adata.obs['source'] = 'in_house'

print(adata_fus)
print(adata_par)
print(adata_res)

AnnData object with n_obs × n_vars = 6044 × 36601
    obs: 'sample', 'source'
    var: 'gene_ids', 'feature_types'
AnnData object with n_obs × n_vars = 10353 × 36601
    obs: 'sample', 'source'
    var: 'gene_ids', 'feature_types'
AnnData object with n_obs × n_vars = 8513 × 36601
    obs: 'sample', 'source'
    var: 'gene_ids', 'feature_types'


In [ ]:
# Concatenate in-house samples first (same gene space).
# Built here (before the HCA load) so its var_names/obs are available to restrict
# the HCA reference before it's ever pulled into memory.
adata_inhouse = ad.concat([adata_fus, adata_par, adata_res], join='outer')
adata_inhouse.var_names_make_unique()

# Placeholders so these columns survive the inner join on .obs below.
# 'cell_type' stays a real label for HCA cells and 'in_house' for ours, so it's
# usable later both as ground truth (HCA) and as a marker of "not yet identified" (ours).
adata_inhouse.obs['cell_type'] = 'in_house'
adata_inhouse.obs['assay'] = "10x (in-house)"  # update to your actual chemistry (e.g. "10x 3' v3") if known

print(adata_inhouse)

## 2. Download HCA Lung Reference via CellxGene Census

Streams lung cells directly — no need to download the full atlas.
Set `census_version` to a pinned release for reproducibility; check available versions at
https://chanzuckerberg.github.io/cellxgene-census/cellxgene_census_docsite_data_release_info.html

In [ ]:
#CENSUS_VERSION = '2025-11-08'  # pin for reproducibility

#census = cellxgene_census.open_soma(census_version=CENSUS_VERSION)

adata_hca = ad.read_h5ad('/mnt/vstor/SOM_CCCC_JGS25/shultesp/data/hca_data/hca_pan_unhealthy_data.h5ad', backed="r")

# Row mask: lung, and is_primary_data drops duplicate cells that Census aggregates
# across multiple submissions of the same underlying dataset.
row_mask = (
    (adata_hca.obs['tissue_general'] == 'lung') & (adata_hca.obs['is_primary_data'])
).values
print(f'Lung + primary cells before subsampling: {row_mask.sum()}')

# Column mask: only genes shared with the in-house data. join='inner' would drop
# the rest later anyway, so there's no reason to carry the full ~61k-gene HCA
# panel into memory just to discard most of it at concat time.
hca_gene_symbols = adata_hca.var['feature_name'].astype(str)
col_mask = hca_gene_symbols.isin(adata_inhouse.var_names).values

# The lung+primary subset is still on the order of 1-3M cells -- far more
# reference cells than needed for scVI batch correction against a ~25k-cell
# in-house cohort, and the direct cause of the earlier MemoryError (the sparse
# concat needs working memory proportional to total nonzero count across both
# matrices). Subsample to a manageable size, stratified by cell_type so rare
# populations aren't lost (they matter later for the nearest-HCA-cell-type
# diagnostics).
N_HCA_TARGET = 50_000

row_idx = np.flatnonzero(row_mask)
meta = pd.DataFrame({
    'idx': row_idx,
    'cell_type': adata_hca.obs['cell_type'].values[row_idx],
})
frac = min(1.0, N_HCA_TARGET / len(meta))


def _stratified_sample(group, frac):
    n = min(len(group), max(1, round(len(group) * frac)))
    return group.sample(n=n, random_state=0)


keep = (
    meta.groupby('cell_type', group_keys=False)
    .apply(lambda g: _stratified_sample(g, frac))
)['idx'].values
keep = np.sort(keep)
print(f'HCA reference cells after stratified subsampling: {len(keep)}')

# Apply both masks together, still backed, then materialize only what's kept
adata_hca = adata_hca[keep, col_mask].to_memory()
adata_hca.var['ensembl_id'] = adata_hca.var_names        # keep original IDs around
adata_hca.var_names = adata_hca.var['feature_name'].astype(str)
adata_hca.var_names_make_unique()
adata_hca.X = adata_hca.X.astype(np.float32, copy=False)


## 3. Verify HCA Raw Counts

scVI requires raw integer counts. CellxGene Census stores raw counts in `.X` by default, but confirm here.

In [4]:
import scipy.sparse as sp

sample_vals = adata_hca.X[:10, :10]
if sp.issparse(sample_vals):
    sample_vals = sample_vals.toarray()

print('Sample HCA X values (should be non-negative integers):')
print(sample_vals)
print('\nAll integer:', np.all(sample_vals == np.floor(sample_vals)))

Sample HCA X values (should be non-negative integers):
[[0. 1. 0. 0. 7. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 1. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 4. 3. 0. 0. 0. 0.]
 [0. 1. 0. 0. 0. 1. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 1. 1. 0. 0. 0. 0.]
 [0. 0. 0. 0. 2. 0. 1. 0. 0. 0.]
 [0. 0. 0. 0. 0. 3. 0. 0. 0. 0.]
 [0. 0. 0. 0. 3. 0. 0. 0. 0. 0.]]

All integer: True


## 4. Tag Metadata — Preserve `cell_type` and `assay` Across the Join

**Fix vs. the original notebook:** `ad.concat(..., join='inner')` inner-joins `.obs`
columns too, not just genes. Because `cell_type` and `assay` only existed on the HCA
side, they were silently dropped during concatenation — which is why the original
"Cell type (HCA annotations)" plot printed `No cell_type column found`.

The fix is to make sure every column we want to keep exists (even as a placeholder)
on **both** sides before concatenating. We also check `assay` here: the Census lung
pull mixes multiple sequencing technologies (e.g. 10x 3' v2/v3, Smart-seq2), and that
was never modeled as a covariate in the original run — it's an unmodeled technical
gap sitting on top of the in-house-vs-HCA batch effect.

In [5]:
adata_hca.obs['sample'] = 'HCA_lung'
adata_hca.obs['source'] = 'public_HCA'

# Make barcodes unique to avoid collisions with in-house barcodes
adata_hca.obs.index = [f'HCA_{bc}' for bc in adata_hca.obs.index]

print('HCA assay breakdown (technical heterogeneity hiding inside the "public_HCA" batch):')
print(adata_hca.obs['assay'].value_counts())

print(adata_hca)

HCA assay breakdown (technical heterogeneity hiding inside the "public_HCA" batch):
assay
10x 3' v2                                   1129888
10x 3' v3                                    948120
10x 5' v2                                    345984
10x 5' v1                                    166485
10x 5' transcription profiling               126024
BD Rhapsody Whole Transcriptome Analysis     122902
GEXSCOPE technology                           58981
10x gene expression flex                      35954
Drop-seq                                      27362
inDrop                                        25652
Smart-seq2                                    15415
10x 3' v1                                     12279
10x 3' transcription profiling                 3857
Name: count, dtype: int64
AnnData object with n_obs × n_vars = 3018903 × 61497
    obs: 'soma_joinid', 'dataset_id', 'assay', 'assay_ontology_term_id', 'cell_type', 'cell_type_ontology_term_id', 'development_stage', 'development_stage

In [ ]:
print(f'Shared genes (HCA restricted to in-house panel): {adata_hca.n_vars}')

adata_combined = ad.concat(
    [adata_inhouse, adata_hca],
    join='inner',   # keeps only genes shared between in-house and HCA
    merge='same',
)

for col in ['source', 'sample', 'assay', 'cell_type']:
    adata_combined.obs[col] = adata_combined.obs[col].astype('category')

print(adata_combined.obs[['source', 'sample', 'assay', 'cell_type']].dtypes)

print(adata_combined)
print(f'\nShared genes: {adata_combined.n_vars}')
print(f'Source breakdown:')
print(adata_combined.obs['source'].value_counts())
print(f'\ncell_type preserved:', 'cell_type' in adata_combined.obs.columns)
print(f'assay preserved:', 'assay' in adata_combined.obs.columns)

## 5. QC Filtering

In [ ]:
sc.pp.filter_cells(adata_combined, min_genes=200)
sc.pp.filter_genes(adata_combined, min_cells=3)

adata_combined.var['mt'] = adata_combined.var_names.str.startswith('MT-')
sc.pp.calculate_qc_metrics(
    adata_combined, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True
)

# Visualize QC metrics split by source before filtering
sc.pl.violin(
    adata_combined,
    ['n_genes_by_counts', 'total_counts', 'pct_counts_mt'],
    groupby='source',
    jitter=0.4,
    multi_panel=True,
)

In [ ]:
adata_combined_filtered = adata_combined[adata_combined.obs.total_counts < 75000, :]
adata_combined_filtered = adata_combined_filtered[adata_combined_filtered.obs.n_genes > 1000, :]
adata_combined_filtered = adata_combined_filtered[adata_combined_filtered.obs.pct_counts_mt < 40, :]

print(adata_combined_filtered)
print(adata_combined_filtered.obs['source'].value_counts())

## 6. Normalize, Log-Transform, and Select HVGs

HVGs are selected with `batch_key='source'` so genes are ranked by variability within each source
independently, then combined. This avoids selecting genes that vary only because of source differences.

In [ ]:
adata_combined_filtered = adata_combined_filtered.copy()  # materialize view
adata_combined_filtered.layers['counts'] = adata_combined_filtered.X.copy()

sc.pp.normalize_total(adata_combined_filtered, target_sum=1e6)
sc.pp.log1p(adata_combined_filtered)
adata_combined_filtered.raw = adata_combined_filtered

sc.pp.highly_variable_genes(
    adata_combined_filtered,
    n_top_genes=3000,
    subset=True,
    layer='counts',
    flavor='seurat_v3',
    batch_key='source',      # rank HVGs within each source independently
)

print(f'HVGs selected: {adata_combined_filtered.n_vars}')

## 7. scVI Setup and Training

- `batch_key='source'`: corrects for in-house vs public technical differences
- `categorical_covariate_keys=['sample', 'assay']`: also accounts for fused/parental/resistant
  variation **and** the sequencing-technology mix inside the HCA pull
- `n_latent=30`: reduced from an earlier 128 — a latent space that large gives the model
  room to preserve batch-specific structure rather than remove it
- `max_epochs` is computed from scvi-tools' own dataset-size heuristic (roughly
  `min(round(20000/n_cells * 400), 400)`) with early stopping, instead of a flat 1000
  chosen without checking convergence

In [ ]:
scvi.model.SCVI.setup_anndata(
    adata_combined_filtered,
    layer='counts',
    batch_key='source',
    categorical_covariate_keys=['sample', 'assay'],
)

In [ ]:
n_obs = adata_combined_filtered.n_obs
heuristic_max_epochs = min(round((20000 / n_obs) * 400), 400)
print(f'n_cells={n_obs}, scvi-tools default max_epochs heuristic ~= {heuristic_max_epochs}')

model = scvi.model.SCVI(adata_combined_filtered, n_latent=30)
model.train(
    accelerator='cpu',  # Apple Silicon GPU
    max_epochs=heuristic_max_epochs,
    early_stopping=True,
    early_stopping_patience=20,
)

In [ ]:
train_elbo = model.history['elbo_train']
val_elbo = model.history['elbo_validation']

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(train_elbo.index, train_elbo.iloc[:, 0], label='train')
ax.plot(val_elbo.index, val_elbo.iloc[:, 0], label='validation')
ax.set_xlabel('epoch')
ax.set_ylabel('ELBO')
ax.legend()
ax.set_title('SCVI training curves — check for plateau / overfitting')
plt.show()

In [ ]:
# Save model so you don't have to retrain
model.save('scvi_HCA_integration_model_v2/', overwrite=True)

## 8. Latent Representation and Clustering

In [ ]:
SCVI_LATENT_KEY = 'X_scVI'
adata_combined_filtered.obsm[SCVI_LATENT_KEY] = model.get_latent_representation()

sc.pp.neighbors(adata_combined_filtered, use_rep=SCVI_LATENT_KEY)

SCVI_CLUSTERS_KEY = 'leiden_scVI'
sc.tl.leiden(
    adata_combined_filtered,
    key_added=SCVI_CLUSTERS_KEY,
    resolution=0.5,
    flavor='igraph',
    n_iterations=2,
    directed=False,
)

sc.tl.umap(adata_combined_filtered, min_dist=0.3)

## 9. Visualization

Key plots to assess integration quality and cell type context.

In [ ]:
# Integration quality: in-house and HCA cells should intermingle if integration worked
sc.pl.umap(
    adata_combined_filtered,
    color=['source'],
    frameon=False,
    title='Source (integration check)',
)

In [ ]:
# In-house sample identity
sc.pl.umap(
    adata_combined_filtered,
    color=['sample'],
    frameon=False,
    groups=['fused', 'parental', 'resistant'],  # exclude HCA_lung from color
    title='In-house samples',
)

In [ ]:
# Leiden clusters
sc.pl.umap(
    adata_combined_filtered,
    color=[SCVI_CLUSTERS_KEY],
    frameon=False,
    legend_loc='on data',
)

In [ ]:
# HCA cell type annotations — shows which lung cell types your cells resemble
sc.pl.umap(
    adata_combined_filtered,
    color=['cell_type'],
    frameon=False,
    legend_fontsize=7,
    title='Cell type (HCA annotations)',
)

## Per-Cell and Per-Cluster Mixing Diagnostics

Two complementary views of "which cells actually overlap":

1. **Per-cell local mixing score** — for every cell, what fraction of its nearest
   neighbors (in the scVI/scANVI latent space) are `public_HCA`? Near the background
   HCA fraction ⇒ well mixed; near 0 ⇒ isolated in its own in-house neighborhood.
2. **Nearest-HCA cell type** — restricted to HCA neighbors only, what `cell_type`
   do they carry? This tells you *which* HCA cell type an in-house cell resembles,
   not just whether it's "near" the reference in general.

In [ ]:
conn = adata_combined_filtered.obsp['connectivities']
is_hca = (adata_combined_filtered.obs['source'] == 'public_HCA').values.astype(float)

neighbor_hca_frac = (
    np.asarray(conn.multiply(is_hca[None, :]).sum(axis=1)).ravel()
    / np.asarray((conn > 0).sum(axis=1)).ravel()
)
adata_combined_filtered.obs['hca_neighbor_frac'] = neighbor_hca_frac

background_hca_frac = is_hca.mean()
print(f'Background HCA fraction (expected value if fully mixed): {background_hca_frac:.3f}')

# A simple, tunable definition of "overlapping": at least as HCA-mixed as random chance
adata_combined_filtered.obs['well_integrated'] = (
    adata_combined_filtered.obs['hca_neighbor_frac'] >= background_hca_frac
)

inhouse_mask = adata_combined_filtered.obs['source'] == 'in_house'
print('\nIn-house cells flagged well_integrated:')
print(adata_combined_filtered.obs.loc[inhouse_mask, 'well_integrated'].value_counts())

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
for s in ['fused', 'parental', 'resistant']:
    vals = adata_combined_filtered.obs.loc[
        adata_combined_filtered.obs['sample'] == s, 'hca_neighbor_frac'
    ]
    ax.hist(vals, bins=40, alpha=0.5, label=s, density=True)
ax.axvline(background_hca_frac, color='k', linestyle='--', label='background HCA fraction')
ax.set_xlabel('Fraction of neighbors that are public_HCA')
ax.set_ylabel('Density')
ax.legend()
ax.set_title('Per-cell local mixing score, by in-house sample')
plt.show()

In [ ]:
sc.pl.umap(
    adata_combined_filtered,
    color=['hca_neighbor_frac'],
    frameon=False,
    color_map='viridis',
    title='Local mixing score (fraction of neighbors from HCA)',
)

In [ ]:
# Cluster-level view: source composition per leiden_scVI cluster
composition = pd.crosstab(
    adata_combined_filtered.obs['leiden_scVI'],
    adata_combined_filtered.obs['source'],
    normalize='index',
)
print('Fraction of each cluster from each source (compare to background HCA fraction above):')
print(composition.sort_values('public_HCA'))

In [ ]:
from sklearn.neighbors import NearestNeighbors

latent = adata_combined_filtered.obsm['X_scVI']
is_hca_mask = (adata_combined_filtered.obs['source'] == 'public_HCA').values
is_inhouse_mask = ~is_hca_mask

nn = NearestNeighbors(n_neighbors=15).fit(latent[is_hca_mask])
_, idx = nn.kneighbors(latent[is_inhouse_mask])

hca_cell_types = adata_combined_filtered.obs.loc[is_hca_mask, 'cell_type'].astype(str).values

# scipy.stats.mode (>=1.11) no longer accepts non-numeric input, so factorize to
# integer codes, take the mode of the codes, then map back to labels.
codes, categories = pd.factorize(hca_cell_types)
nn_codes = codes[idx]  # shape: (n_inhouse_cells, 15)

maj = mode(nn_codes, axis=1, keepdims=False)

inhouse_names = adata_combined_filtered.obs_names[is_inhouse_mask]
adata_combined_filtered.obs.loc[inhouse_names, 'nearest_hca_cell_type'] = categories[maj.mode]
adata_combined_filtered.obs.loc[inhouse_names, 'nearest_hca_cell_type_agreement'] = maj.count / nn_codes.shape[1]

print('Which HCA cell types do well-integrated in-house cells resemble?')
well_mixed_inhouse = inhouse_names[
    adata_combined_filtered.obs.loc[inhouse_names, 'well_integrated'].values
]
print(
    adata_combined_filtered.obs.loc[well_mixed_inhouse, 'nearest_hca_cell_type']
    .value_counts()
    .head(15)
)

print('Which HCA cell types do in-house cells resemble (regardless of mixing score)?')
print(
    adata_combined_filtered.obs.loc[inhouse_names, 'nearest_hca_cell_type']
    .value_counts()
    .head(15)
)

print('\nMean agreement among the top-15 nearest neighbors (higher = more confident match):')
print(
    adata_combined_filtered.obs.loc[inhouse_names]
    .groupby('nearest_hca_cell_type', observed=True)['nearest_hca_cell_type_agreement']
    .mean()
    .sort_values(ascending=False)
    .head(15)
)

## 10. Differential Expression — In-House Samples Only

Restrict DE to in-house cells so we compare fused/parental/resistant against each other,
not against HCA lung cells.

In [ ]:
adata_inhouse_filtered = adata_combined_filtered[
    adata_combined_filtered.obs['source'] == 'in_house'
].copy()

print(adata_inhouse_filtered)
print(adata_inhouse_filtered.obs['sample'].value_counts())

In [ ]:
# Re-load the model and get latent for in-house subset only
# (model.differential_expression can subset internally via indices)
de_df = model.differential_expression(
    adata=adata_inhouse_filtered,
    groupby='sample',
    mode='change',
)
de_df.head()

In [ ]:
markers = {}
cats = {'fused', 'resistant', 'parental'}
for c in cats:
    cid = f'{c} vs Rest'
    cell_type_df = de_df.loc[de_df.comparison == cid]
    cell_type_df = cell_type_df[cell_type_df.lfc_mean > 0]
    cell_type_df = cell_type_df[cell_type_df['bayes_factor'] > 3]
    cell_type_df = cell_type_df[cell_type_df['non_zeros_proportion1'] > 0.1]
    markers[c] = cell_type_df.index.tolist()[:3]

print('Top markers per sample:', markers)

In [ ]:
sc.tl.dendrogram(adata_inhouse_filtered, groupby='sample', use_rep='X_scVI')

sc.pl.dotplot(
    adata_inhouse_filtered,
    markers,
    groupby='sample',
    dendrogram=True,
    color_map='Blues',
    swap_axes=True,
    use_raw=True,
    standard_scale='var',
)

## 11. Cell Type Composition of In-House Samples

Shows what fraction of each in-house sample maps to each HCA-annotated cluster.

In [ ]:
composition = (
    adata_inhouse_filtered.obs
    .groupby(['sample', 'nearest_hca_cell_type'], observed=True)
    .size()
    .unstack(fill_value=0)
)
composition_norm = composition.div(composition.sum(axis=1), axis=0)

composition_norm.T.plot(
    kind='bar',
    stacked=False,
    figsize=(14, 5),
    title='Nearest-HCA cell type composition by in-house sample',
)
plt.ylabel('Fraction of cells')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 12. UMAP — In-House Cells Only

Plot just in-house cells on the shared latent UMAP to see where fused/parental/resistant land
relative to the HCA-defined clusters, and how well-integrated each one is.

In [ ]:
sc.pl.umap(
    adata_inhouse_filtered,
    color=['sample', SCVI_CLUSTERS_KEY, 'hca_neighbor_frac', 'well_integrated'],
    frameon=False,
    ncols=2,
)